# ⚡ VisionForge — Model Training Studio
> **YOLOv8 × Roboflow** — Train custom object detection models on Google Colab T4 GPU

---
### 📋 Steps
1. **Connect T4 GPU** — verify with `nvidia-smi`
2. **Install dependencies**
3. **Download dataset** from Roboflow
4. **Train YOLOv8** with your hyperparameters
5. **Launch VisionForge UI** via `ngrok` tunnel
6. **View results** — weights, metrics, inference

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

---
## 🖥️ Step 1 — Connect T4 GPU & Verify

In [ ]:
# ── Verify T4 GPU is connected ────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Double-check PyTorch sees the GPU ─────────────────────────────────────────
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU")

---
## 📦 Step 2 — Install Dependencies

In [ ]:
# ── Install all required packages ─────────────────────────────────────────────
!pip install ultralytics roboflow flask flask-cors pyngrok -q
print("✅ All packages installed.")

---
## ⚙️ Step 3 — Configuration
Fill in your Roboflow credentials and training hyperparameters below.

In [ ]:
# ── 🔧 EDIT THESE VALUES ──────────────────────────────────────────────────────

ROBOFLOW_API_KEY  = "YOUR_API_KEY_HERE"     # Your Roboflow API key
WORKSPACE         = "your-workspace"        # Roboflow workspace slug
PROJECT_NAME      = "your-project-name"     # Roboflow project slug
VERSION_NUM       = 1                       # Dataset version number

# Training hyperparameters
MODEL_VARIANT     = "yolov8n.pt"           # yolov8n / yolov8s / yolov8m / yolov8l / yolov8x
EPOCHS            = 25
IMAGE_SIZE        = 640
BATCH_SIZE        = 16

# ── Ngrok auth token (get free at https://ngrok.com) ─────────────────────────
NGROK_AUTH_TOKEN  = "YOUR_NGROK_TOKEN_HERE"

print("✅ Configuration set.")
print(f"   Project  : {PROJECT_NAME} (v{VERSION_NUM})")
print(f"   Model    : {MODEL_VARIANT}")
print(f"   Epochs   : {EPOCHS}  |  Image size: {IMAGE_SIZE}  |  Batch: {BATCH_SIZE}")

---
## 🗂️ Step 4 — Download Dataset from Roboflow

In [ ]:
# ── Download dataset ──────────────────────────────────────────────────────────
from roboflow import Roboflow

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
version = project.version(VERSION_NUM)
dataset = version.download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"
print(f"✅ Dataset ready at: {dataset.location}")
print(f"   data.yaml: {DATA_YAML}")

---
## 🏋️ Step 5 — Train YOLOv8

In [ ]:
# ── Train the model ───────────────────────────────────────────────────────────
from ultralytics import YOLO

model   = YOLO(MODEL_VARIANT)
results = model.train(
    data      = DATA_YAML,
    epochs    = EPOCHS,
    imgsz     = IMAGE_SIZE,
    batch     = BATCH_SIZE,
    device    = 0,           # 0 = first GPU (T4)
    verbose   = True,
    project   = "visionforge_runs",
    name      = f"{PROJECT_NAME}_v{VERSION_NUM}",
)

print("\n✅ Training complete!")
print(f"   Best weights: {results.save_dir}/weights/best.pt")

---
## 📊 Step 6 — View Results

In [ ]:
# ── Display training curves ───────────────────────────────────────────────────
from IPython.display import Image, display
import glob, os

result_imgs = sorted(
    glob.glob("visionforge_runs/**/results.png", recursive=True),
    key=os.path.getmtime, reverse=True
)

if result_imgs:
    print(f"📈 Results chart: {result_imgs[0]}")
    display(Image(result_imgs[0], width=900))
else:
    print("⚠️  No results.png found yet.")

In [ ]:
# ── Run validation on best weights ───────────────────────────────────────────
import glob, os
from ultralytics import YOLO

best_weights = sorted(
    glob.glob("visionforge_runs/**/weights/best.pt", recursive=True),
    key=os.path.getmtime, reverse=True
)

if best_weights:
    print(f"🚀 Loading: {best_weights[0]}")
    best_model = YOLO(best_weights[0])
    metrics    = best_model.val(data=DATA_YAML)
    print(f"\n📊 mAP50   : {metrics.box.map50:.4f}")
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   Precision: {metrics.box.mp:.4f}")
    print(f"   Recall   : {metrics.box.mr:.4f}")
else:
    print("⚠️  No best.pt found. Run training first.")

---
## 💾 Step 7 — Export Model

In [ ]:
# ── Export to ONNX / TorchScript / TFLite ────────────────────────────────────
# Options: 'onnx', 'torchscript', 'tflite', 'coreml', 'engine' (TensorRT)

EXPORT_FORMAT = "onnx"  # Change as needed

if best_weights:
    export_model = YOLO(best_weights[0])
    export_path  = export_model.export(format=EXPORT_FORMAT)
    print(f"✅ Exported to: {export_path}")
else:
    print("⚠️  No best.pt found. Run training first.")

---
## 🌐 Step 8 — Launch VisionForge Web UI (Optional)
Starts the Flask server and opens it via an ngrok tunnel.

In [ ]:
# ── Upload and launch the VisionForge Flask app ───────────────────────────────
# Make sure you have uploaded app.py and templates/index.html to Colab

import threading, time
from pyngrok import ngrok, conf

# Set ngrok auth token
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Start Flask in a background thread
def run_flask():
    import subprocess
    subprocess.run(["python", "app.py"])

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)  # Wait for Flask to boot

# Open tunnel
tunnel = ngrok.connect(5000)
print(f"\n🌐 VisionForge UI → {tunnel.public_url}")
print("   Open the link above in your browser.")

---
## ⬇️ Step 9 — Download Your Weights

In [ ]:
# ── Download best.pt to your machine ─────────────────────────────────────────
from google.colab import files
import glob, os

best_weights = sorted(
    glob.glob("visionforge_runs/**/weights/best.pt", recursive=True),
    key=os.path.getmtime, reverse=True
)

if best_weights:
    print(f"⬇️  Downloading: {best_weights[0]}")
    files.download(best_weights[0])
else:
    print("⚠️  No best.pt found. Run training first.")